In [ ]:
# ============================================
# INSTALL LIBRARIES
# ============================================
!pip install pandas nltk scikit-learn matplotlib seaborn

# ============================================
# IMPORT LIBRARIES
# ============================================
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.decomposition import PCA

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

# ============================================
# LOAD DATASET
# ============================================
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename, encoding='latin-1')

df.columns = df.columns.str.strip().str.lower()

text_col = None
for col in df.columns:
    if 'response' in col or 'feedback' in col or 'text' in col:
        text_col = col
        break

print("Detected text column:", text_col)

# ============================================
# PREPROCESSING
# ============================================
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)

    words = text.split()
    words = [
        lemmatizer.lemmatize(w)
        for w in words
        if w not in stop_words and len(w) > 2
    ]
    return " ".join(words)

df['cleaned'] = df[text_col].apply(clean_text)

# ============================================
# LABELING
# ============================================
keywords = {
    'Appreciation': ['thank','grateful','appreciate','good','great','satisfied'],
    'Challenges': ['problem','issue','delay','error','slow','difficult'],
    'Suggestion': ['should','recommend','improve','better','suggest']
}

def assign_labels(text):
    labels = []
    for label, words in keywords.items():
        if any(w in text for w in words):
            labels.append(label)
    return labels if labels else ['Neutral']

df['labels'] = df['cleaned'].apply(assign_labels)

# ============================================
# FEATURE EXTRACTION
# ============================================
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X = vectorizer.fit_transform(df['cleaned'])

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['labels'])

# ============================================
# TRAIN TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================
# MODEL
# ============================================
model = OneVsRestClassifier(
    SVC(kernel='linear', probability=True)
)

model.fit(X_train, y_train)

# ============================================
# EVALUATION
# ============================================
y_pred_prob = model.predict_proba(X_test)

threshold = 0.3
y_pred = (y_pred_prob >= threshold).astype(int)

print("F1 Micro:", f1_score(y_test, y_pred, average='micro'))
print("F1 Macro:", f1_score(y_test, y_pred, average='macro'))

print(classification_report(y_test, y_pred, target_names=mlb.classes_))

# ============================================
# VISUALIZATION (PCA)
# ============================================
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X.toarray())

svm_vis = SVC(kernel='linear')
svm_vis.fit(X_2d, y.argmax(axis=1))

def plot_svm():
    plt.figure(figsize=(10,6))

    x_min, x_max = X_2d[:,0].min()-1, X_2d[:,0].max()+1
    y_min, y_max = X_2d[:,1].min()-1, X_2d[:,1].max()+1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    Z = svm_vis.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.2)

    scatter = plt.scatter(
        X_2d[:,0], X_2d[:,1],
        c=y.argmax(axis=1),
        cmap='rainbow',
        edgecolor='k'
    )

    plt.title("SVM Hyperplane Visualization")
    plt.legend(*scatter.legend_elements())
    plt.show()

plot_svm()

print("Support vectors per class:")
print(svm_vis.n_support_)

# ============================================
# PREDICTION
# ============================================
def predict(text, threshold=0.3):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])

    probs = model.predict_proba(vec)[0]

    labels = [
        mlb.classes_[i]
        for i, p in enumerate(probs)
        if p >= threshold
    ]

    if not labels:
        labels = ["Neutral"]

    return labels, probs

def show_percentages(probs):
    for label, prob in zip(mlb.classes_, probs):
        print(f"{label}: {prob*100:.2f}%")

while True:
    x = input("Enter text (or exit): ")
    if x.lower() == 'exit':
        break

    labels, probs = predict(x)

    print("Predicted Labels:", labels)
    show_percentages(probs)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Saving UAQTEresponses.csv to UAQTEresponses.csv
Detected text column: response
